# Module 5.8 — Lab: Cross-Dataset Hands-On

**Module 5: Real Project — ML for Animal Adoption | Week 2 | Lab (Session 5.8)**

---

In 5.7 you saw the cross-dataset story laid out: how the PetFinder model fails on AAC, why pooling doesn't help, what the multi-task architecture actually does, and the trunk-LR result that beats the §12 baseline. Now you build it yourself.

By the end of this lab you'll have a working feel for:

- How to verify a cross-dataset transfer is failing (and what "failure" looks like in the numbers)
- How the shared-trunk + two-heads architecture is wired up — including the part where each head takes its own dataset's specific features
- How to extract the trunk's learned representation and probe it with a linear classifier on a new domain's labels
- Why the trunk transfers across shelters but the heads don't (a real architectural lesson, not a quirk)
- (Stretch) Why pretrain-then-finetune doesn't extract more signal than AAC-only training on the shared feature space

You won't write the multi-task model from scratch — the class shell is given. You'll fill in `forward_pf` / `forward_aac` and the training loop. Same pattern as the M05-lab3 cascade hands-on (toolkit gives you the bones, you exercise the calls).


### Glossary

- **Multi-task model** — one network that learns to predict multiple things at once. In this lab: predict adoption on PetFinder AND adoption on AAC, using a single shared trunk plus separate output heads.
- **Shared trunk** — the layers of the network that both tasks pass through. Forced to learn features useful for both — which is what makes them transfer.
- **Dataset-specific head** — the layers after the trunk that produce a per-dataset prediction. Each head also gets to use its own dataset's exclusive features (PetFinder's `PhotoAmt`, AAC's `intake_stray`, etc.).
- **Linear probe** — a tiny model (logistic regression, here) trained on top of a frozen learned representation. Used to ask: "does this representation contain signal for *this* new task?"
- **Trunk transfer** — the phenomenon you'll see in Task 3 — the trunk's representation transfers across domains better than the heads do.
- **Pretrain-finetune** — train on a big source domain, then continue training (usually at a low learning rate) on a small slice of the target domain. The "standard" domain-adaptation move when source and target are similar. Spoiler: doesn't help here, and Task 5 (stretch) explores why.

**Quick recap from 5.7** — PetFinder and AAC have different *signal sources*. PetFinder's signal lives in animal demographics (age, breed, photos). AAC's signal lives in shelter-process features (intake_type, intake_condition). The 6 features they share aren't enough to ship a model cross-shelter. The multi-task framework's job is to learn what's *common* despite that mismatch.


In [ ]:
import os
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


## Setup — download both bundles, load PetFinder and AAC

This lab uses **two** data bundles:

- `petfinder-mini.zip` — same one you've used in 5.2 / 5.4 / 5.6 / lab5 / lab6
- `aac-mini.zip` — Austin Animal Center subset introduced in 5.7

Your instructor will give you a Google Drive URL for each. Replace the `CONFIGURE_ME` strings below.


In [ ]:
PF_URL  = 'CONFIGURE_ME'     # ← your instructor will give you the petfinder-mini URL
AAC_URL = 'CONFIGURE_ME'     # ← your instructor will give you the aac-mini URL
PF_DIR  = '/content/petfinder'
AAC_DIR = '/content/aac'


def fetch(url, target_dir, marker_file):
    if os.path.exists(f'{target_dir}/{marker_file}'):
        return
    if url == 'CONFIGURE_ME':
        raise RuntimeError(f"Set the URL for {target_dir} above.")
    os.makedirs(target_dir, exist_ok=True)
    import gdown
    zip_path = f'{target_dir}/bundle.zip'
    gdown.download(url=url, output=zip_path, quiet=False, fuzzy=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(target_dir)
    os.remove(zip_path)


fetch(PF_URL,  PF_DIR,  'train.csv')
fetch(AAC_URL, AAC_DIR, 'aac_prep.py')

# Toolkit ships in petfinder-mini.zip; aac_prep.py ships in aac-mini.zip.
import sys
sys.path.insert(0, PF_DIR)
sys.path.insert(0, AAC_DIR)
from m05_labbench import _encode_features, _MultiHeadMLP, _make_layers, LabBench
from aac_prep import load_aac

# --- PetFinder ---
pf = pd.read_csv(f'{PF_DIR}/train.csv')
pf['adopted_in_30_days'] = (pf['AdoptionSpeed'] <= 2).astype(int)
pf['age_adult']      = (pf['Age'] >= 12).astype(int)
pf['is_young']       = (pf['Age'] < 6).astype(int)
pf['is_mixed_breed'] = ((pf['Breed1'] == 307) | (pf['Breed2'] != 0)).astype(int)
pf['gender_male']    = (pf['Gender'] == 1).astype(int)
pf['gender_female']  = (pf['Gender'] == 2).astype(int)
pf['sterilized_yes'] = (pf['Sterilized'] == 1).astype(int)

# --- AAC ---
aac = load_aac()
# Per-species median-cutoff target (T_AAC from Session 16: dogs 7.21d, cats 25.82d).
T_AAC = {1: 7.21, 2: 25.82}
fast = np.zeros(len(aac), dtype=int)
for sid, T in T_AAC.items():
    m = (aac['Type'] == sid)
    fast[m & (aac['outcome_type'] == 'Adoption') & (aac['days_in_shelter'] <= T)] = 1
aac['adopted_fast_AAC'] = fast

print(f'PetFinder: {len(pf):,} rows  (dogs={(pf["Type"]==1).sum():,}, cats={(pf["Type"]==2).sum():,})')
print(f'AAC:       {len(aac):,} rows  (dogs={(aac["Type"]==1).sum():,}, cats={(aac["Type"]==2).sum():,})')

# Feature blocks
SHARED_FEATURES = [
    ('age_adult',      'as_is'),
    ('gender_male',    'as_is'),
    ('gender_female',  'as_is'),
    ('sterilized_yes', 'as_is'),
    ('is_mixed_breed', 'as_is'),
    ('is_young',       'as_is'),
]
PF_SPECIFIC = [
    ('MaturitySize', ('ordinal', (1, 3))),
    ('FurLength',    ('ordinal', (1, 3))),
    ('Vaccinated',   ('binary', 'eq', 1)),
    ('Dewormed',     ('binary', 'eq', 1)),
    ('Health',       ('ordinal', (1, 3))),
    ('PhotoAmt',     'as_is'),
    ('Fee',          'as_is'),
]
AAC_SPECIFIC = [
    ('intake_stray',                'as_is'),
    ('intake_owner_surrender',      'as_is'),
    ('intake_public_assist',        'as_is'),
    ('intake_euthanasia_request',   'as_is'),
    ('cond_normal',                 'as_is'),
    ('cond_injured',                'as_is'),
    ('cond_sick',                   'as_is'),
    ('cond_nursing',                'as_is'),
    ('had_name',                    'as_is'),
    ('intake_month',                'as_is'),
]
DOGS = lambda d: d['Type'] == 1
CATS = lambda d: d['Type'] == 2
SEED = 42


---

### Task 1: Reproduce the direct-transfer failure

The first move in any cross-dataset story is to verify that the naive transfer actually fails. **Train a single MLP on PetFinder dogs (target `adopted_in_30_days`) and apply it directly to AAC dogs (target `adopted_fast_AAC`).** Compute AUC on both: same-dataset (sanity check) and cross-dataset (the failure you're verifying).

Suggested structure:

```python
# 1. Subset to dogs on both datasets, encode SHARED features on both.
pf_dogs  = pf[DOGS(pf)].copy()
aac_dogs = aac[DOGS(aac)].copy()
pf_X,  _ = _encode_features(pf_dogs,  SHARED_FEATURES)
aac_X, _ = _encode_features(aac_dogs, SHARED_FEATURES)
pf_y  = pf_dogs['adopted_in_30_days'].values.astype(int)
aac_y = aac_dogs['adopted_fast_AAC'].values.astype(int)

# 2. Split PF into 75% train / 25% test. Fit the StandardScaler on PF train —
#    apply that SAME scaler to AAC so they share a coordinate system.
pf_tr_idx, pf_te_idx = train_test_split(
    np.arange(len(pf_X)), test_size=0.25, random_state=SEED, stratify=pf_y,
)
# ... fit scaler, transform pf_tr / pf_te / aac

# 3. Train a single-headed MLP via _MultiHeadMLP(n_aux=0, shared_hidden=(64,32)).
#    Use LabBench._train for 100 epochs at lr=1e-3 (full-batch, no aux losses).

# 4. Compute AUC on PF test (sanity) and on AAC (cross).
def auc_of(m, X, y):
    m.eval()
    with torch.no_grad():
        p = torch.sigmoid(m(torch.tensor(X, dtype=torch.float32, device=DEVICE))[0]).cpu().numpy()
    return roc_auc_score(y, p)
```

**Expected** (Session 16 numbers, ±0.02 seed variance is normal):

- PF→PF same-dataset AUC: roughly **0.67** for dogs
- PF→AAC cross-dataset AUC: roughly **0.60** for dogs (§12 baseline was 0.595)

The cross AUC is technically above 0.5 — there's *some* rank signal — but well below same-dataset. **This is the number Task 3 will try to beat by +0.05 or more.**


In [ ]:
# Task 1: your code here


---

### Task 2: Build and train the multi-task model

The architecture from 5.7 cell 18:

```
SHARED features (6 cols) ────► Trunk(64 → 32) ────► trunk_out (32d)
                                                          │
                                                          ├─ ++ PF-specific (7 cols)  ────► PF head  ────► P(fast | PF)
                                                          └─ ++ AAC-specific (10 cols) ────► AAC head ────► P(fast | AAC)
```

The class shell is given below. **Your job:**

1. Fill in `forward_pf` and `forward_aac` (one line each).
2. Write a training loop that does **a full-batch step on both datasets every epoch** with `loss = 0.5 * BCE_pf + 0.5 * BCE_aac`.

Use the same `(64, 32)` trunk + small `(16,)` head pattern. Train for **100 epochs** at `lr=1e-3` on **dogs only** (use `DOGS` filter).

You need to encode three blocks for each dataset:

- SHARED features → goes into the trunk
- PF_SPECIFIC features → goes into the PF head alongside the trunk output
- AAC_SPECIFIC features → goes into the AAC head alongside the trunk output

**Scaler tip:** fit one StandardScaler on `np.vstack([pf_train_shared, aac_train_shared])` so the trunk sees a single coordinate system. Fit a separate scaler per dataset on the specific blocks.

After training, evaluate same-dataset AUC on each dataset's held-out test set. (Task 3 will do the interesting cross-evaluation.)

**Expected same-dataset AUCs** (Session 16 numbers, ±0.02 seed variance):

- PF dogs same-dataset AUC: ~**0.67** (matches what Task 1 got — sanity)
- AAC dogs same-dataset AUC: ~**0.71** (higher than PF — AAC has more data + the AAC-specific intake features carry real signal)


In [ ]:
# Task 2: build the multi-task model and train it.

class MultiDatasetMLP(nn.Module):
    """Shared trunk + per-dataset heads with dataset-specific feature blocks."""

    def __init__(self, n_shared, n_pf, n_aac,
                 shared_hidden=(64, 32), head_hidden=(16,), dropout=0.2):
        super().__init__()
        trunk_layers, trunk_out = _make_layers(n_shared, shared_hidden, dropout)
        self.trunk = nn.Sequential(*trunk_layers)
        self.trunk_out_dim = trunk_out

        pf_layers, pf_h = _make_layers(trunk_out + n_pf, head_hidden, dropout)
        pf_layers.append(nn.Linear(pf_h, 1))
        self.head_pf = nn.Sequential(*pf_layers)

        aac_layers, aac_h = _make_layers(trunk_out + n_aac, head_hidden, dropout)
        aac_layers.append(nn.Linear(aac_h, 1))
        self.head_aac = nn.Sequential(*aac_layers)

    def forward_pf(self, x_shared, x_pf):
        # TODO (1 line): apply self.trunk to x_shared, concatenate x_pf onto the
        # trunk output, pass through self.head_pf. Return shape (batch,) —
        # remember to .squeeze(-1) the final linear-layer output.
        ...

    def forward_aac(self, x_shared, x_aac):
        # TODO (1 line, symmetric to forward_pf using self.head_aac)
        ...

    def trunk_only(self, x_shared):
        return self.trunk(x_shared)


# --- Training-data prep ---------------------------------------------------
# Encode SHARED + PF_SPECIFIC for PF dogs, SHARED + AAC_SPECIFIC for AAC dogs.
# 75/25 split per dataset, stratified by the binary target. Save the test
# splits — you'll re-use them in Task 3.

# --- Train ---------------------------------------------------------------
# Instantiate MultiDatasetMLP(n_shared=6, n_pf=7, n_aac=10).
# For 100 epochs, full-batch:
#   - p_pf  = model.forward_pf(pf_train_shared,  pf_train_specific)
#   - p_aac = model.forward_aac(aac_train_shared, aac_train_specific)
#   - loss  = 0.5 * BCEWithLogitsLoss()(p_pf,  pf_y_train)  +
#             0.5 * BCEWithLogitsLoss()(p_aac, aac_y_train)
#   - opt.zero_grad(); loss.backward(); opt.step()

# --- Eval same-dataset AUC ----------------------------------------------
# Run forward_pf on PF test, forward_aac on AAC test, compute AUC for each.


---

### Task 3: Trunk transfer — the linear probe

Same model from Task 2. Now use only the **trunk**. Push every AAC train row through `model.trunk_only(x_shared_scaled)` to get its 32-d representation. Train a logistic regression (the **linear probe**) on AAC's training labels. Score on AAC's test set.

```python
# 1. Extract trunk representations for AAC train + AAC test.
#    Use the SHARED scaler you fit in Task 2 so the trunk sees the same
#    distribution it was trained on.
model.eval()
with torch.no_grad():
    trunk_aac_tr = model.trunk_only(torch.tensor(aac_tr_shared_s, dtype=torch.float32, device=DEVICE)).cpu().numpy()
    trunk_aac_te = model.trunk_only(torch.tensor(aac_te_shared_s, dtype=torch.float32, device=DEVICE)).cpu().numpy()

# 2. Train LR on the trunk representation with AAC labels.
lr = LogisticRegression(max_iter=1000).fit(trunk_aac_tr, aac_y_tr)

# 3. Score on AAC test set.
trunk_auc_aac = roc_auc_score(aac_y_te, lr.predict_proba(trunk_aac_te)[:, 1])
print(f'Trunk + LR on AAC labels (dogs): AUC {trunk_auc_aac:.3f}')
```

**Expected:** AUC roughly **0.66** on dogs (Session 16 number was 0.661). Compare to:

- §12 baseline (PF-only single-task → AAC, AUC 0.595 from M05-4 cell 19): your trunk-LR should beat this by **+0.05 or more** — the "real signal" threshold.
- Task 1's cross-dataset direct-transfer AUC (~0.60): the trunk-LR beats the bare PF model's transfer by ~+0.06.

**That gap is the headline of this lab.** The same model whose head cross-transferred poorly (Task 1) has a **trunk** whose representation transfers usefully (Task 3).

**Optional repeat on cats.** If you want to verify the cats result too (Session 16: 0.630, +0.056 over §12), repeat Tasks 2 + 3 with `CATS` instead of `DOGS`. Cats' multi-task trunk-LR gain is a hair smaller than dogs' but lands above the same threshold.


In [ ]:
# Task 3: your code here


---

### Task 4: Interpretation — why does the trunk transfer but the head doesn't?

No code. Read your Task 1 / Task 2 / Task 3 numbers and write down (for the writeup at the end):

1. The PF head was trained with `trunk_out` AND `x_pf` as inputs. What does it use those PF-specific features for? When you apply the PF head to an AAC row, you have to zero-pad the PF-specific block (AAC rows don't have `PhotoAmt` etc.). **What happens to the head's reasoning when those inputs are forced to zero?**
2. The trunk only ever saw the 6 SHARED features as input. Its gradient had to simultaneously help **both** heads do well — on **both** datasets' labels. What's the only kind of feature the trunk could rely on, given that constraint?
3. In your own words: **why** is the trunk's representation more transferable than the head's?

This is the architectural finding the lecture summarized as **"shared trunk learns invariance, heads memorize."** Being able to say *why* in your own words is the whole point of this lab.


---

### Task 5 (stretch): The pretrain-finetune null result

This replicates Session 16's Q2 stop-condition. Train a **single-headed MLP** (use `_MultiHeadMLP(n_in, n_aux=0, ...)` from the toolkit, same as Task 1) in two stages:

1. **Pretrain** on full PetFinder dogs train set (`adopted_in_30_days`), 100 epochs at `lr=1e-3`.
2. **Finetune** on a 1,000-row AAC slice (target `adopted_fast_AAC`), **30 epochs at `lr=1e-4`** — the lower LR is critical so you don't catastrophically forget the PetFinder representation.

Score on the AAC test set (the ~21K rows that aren't in the 1,000-row slice).

Then train a second model — **AAC-only**, same single-headed MLP, same 1,000-row slice, 100 epochs at `lr=1e-3` — and score on the same AAC test set.

```python
# Pretrain a fresh single-headed MLP on PetFinder dogs.
# (Use LabBench._train, batch_size=None for full-batch.)

# Finetune the SAME model on a 1000-row AAC slice at lr=1e-4 for 30 epochs.

# Train a second AAC-only model on just the 1000-row slice (100 epochs, lr=1e-3).

# Compare AUC on AAC test for: pretrain-only, pretrain-finetune, AAC-only-on-1K.
```

**Expected** (Session 16):

- Pretrain-only AUC on AAC: ~**0.61** for dogs (matches §12)
- Pretrain-finetune AUC: ~**0.61** for dogs (essentially unchanged)
- AAC-only on 1,000 rows: ~**0.65** for dogs (beats both!)

That's the **null result**: pretrain-finetune doesn't help over pretrain-only, and 1,000 AAC labels alone are worth more than all the PetFinder data combined with that same 1,000 AAC slice. When source and target have a *signal-source mismatch* (PF demographics vs AAC intake-circumstances), domain-adaptation methods that operate on the shared 6-feature subset can't extract more signal than training directly on target-domain labels. The bottleneck is feature-space coverage, not training procedure.

That's why the investigation's Q2 "stop condition" triggered — at this point you don't escalate to fancier methods (DANN, importance reweighting). You go get more target-domain labels.


In [ ]:
# Task 5 (stretch): your code here


---

## Writeup

In a markdown cell at the bottom, write **5–7 bullets** answering:

- (Task 1) What's the AUC for PF→AAC direct transfer on dogs? Roughly how much below the same-dataset PF→PF AUC?
- (Task 2) What same-dataset AUCs did your multi-task model land at for PF and AAC dogs?
- (Task 3) What's your trunk-LR AUC on AAC dogs? How does that compare to (a) §12's PF-only baseline (0.595) and (b) Task 1's direct-transfer number?
- (Task 4) Answer your three Task 4 questions in your own words. The key sentence: **why** does the trunk transfer where the head doesn't?
- (Task 5, optional) AUC for pretrain-only / pretrain-finetune / AAC-only-on-1K. Does pretrain-finetune beat AAC-only-on-1K? What does that mean about source-domain data value?
- One sentence: what's the headline architectural lesson — what does the multi-task framework actually *buy* you, given that the heads themselves don't transfer?


---

## Summary

- Reproduced the PetFinder→AAC direct-transfer failure — same-dataset baseline, cross-dataset gap.
- Built the multi-task model end-to-end — shared trunk + two dataset-specific heads — and trained it with a balanced two-loss objective.
- Extracted the trunk's representation and verified the **+0.05+ trunk-LR finding** on AAC labels.
- Articulated *why* the trunk transfers and the heads don't — invariance vs memorization.
- (Stretch) Saw for yourself that pretrain-finetune doesn't help when source and target have a signal-source mismatch.

You now have a working multi-task model and a working linear probe. Both are tools you'll see again — multi-task is the M06 deployment recipe for Taiwan, and linear probes are how interpretability work (M04 deep dive style) gets done on real production representations.

---

**Next session (Lecture):** 5.9 — The PetFinder scandal. We pivot from architecture back to the data ethics that ML papers learned the hard way.
